# Gold Layer - Product Pairs Fact Table

## Purpose
Identify frequently co-purchased product pairs for recommendation systems, cross-selling strategies, and inventory optimization.

## Type
**Fact Table** (materialized, 42M rows)

## Input
* **Source:** `big_data.silver.order_products` (33.8M rows)

## Output
* **Target:** `big_data.gold.ft_product_pairs`
* **Rows:** ~42M product pairs (filtered by min 5 co-purchases)
* **Primary Key:** (product_id_1, product_id_2)

## Use Cases
* 🛒 Recommendation systems ("Customers who bought X also bought Y")
* 💰 Cross-selling campaigns
* 📦 Inventory planning (complementary products)
* 🏪 Store layout optimization

## Transformations

### Step 1: Self-Join to Build Product Pairs
* Self-join `order_products` on order_id
* Filter to ensure product_id_1 < product_id_2 (avoid duplicates)
* Select order_id and both product IDs

### Step 2: Aggregate and Filter
* GROUP BY (product_id_1, product_id_2)
* COUNT occurrences as `times_bought_together`
* Filter pairs with >= 5 co-purchases (remove noise)
* Add `_gold_timestamp`

## Data Quality Validations

### Technical Validations
* Row count > 1M (expected ~42M)
* NOT NULL on product_id_1, product_id_2, times_bought_together
* All times_bought_together >= 5 (filter threshold)

### Business Validations
* product_id_1 < product_id_2 (no reversed duplicates)
* Times bought together distribution is reasonable

## Why Materialized Table (not View)?
* ❌ Self-join on 33.8M rows is VERY expensive
* ✅ Result used frequently (dashboards, APIs)
* ✅ 42M rows too large for on-demand computation
* ✅ Can refresh daily (not real-time)

## Persistence
Only persists to Delta table if **all validations pass**.

## Execution
Run all cells sequentially. Expected runtime: ~8-12 minutes.

In [0]:
%run ../UTILS/data_quality_checks

In [0]:
# PySpark imports
from pyspark.sql import functions as F

In [0]:
# Schema configuration
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Source table
source_table = "order_products"

# Target table (fact table with ft_ prefix)
target_table = "ft_product_pairs"

# Validation thresholds
expected_metrics = {
    "min_pairs": 1_000_000,  # Expect at least 1M pairs
    "min_times_together": 5   # Filter threshold
}

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Configuration:")
print(f"  Source schema: {silver_schema}")
print(f"  Source table: {source_table}")
print(f"  Target: {gold_schema}.{target_table}")

In [0]:
print("Step 1: Building product pairs via self-join...")

# Load order_products
order_products = spark.table(f"{silver_schema}.{source_table}")

print(f"  Order-Products loaded: {order_products.count():,} rows")

# Self-join to find product pairs in the same order
# Filter: product_id_1 < product_id_2 to avoid duplicate pairs (A,B) and (B,A)
pairs = order_products.alias("a").join(
    order_products.alias("b"),
    (F.col("a.order_id") == F.col("b.order_id")) & 
    (F.col("a.product_id") < F.col("b.product_id"))
).select(
    F.col("a.product_id").alias("product_id_1"),
    F.col("b.product_id").alias("product_id_2"),
    F.col("a.order_id")
)

print(f"  Product pairs identified: {pairs.count():,} raw pairs")

In [0]:
print("Step 2: Aggregating and filtering product pairs...")

# Aggregate by product pair and count occurrences
product_pairs_gold = pairs.groupBy("product_id_1", "product_id_2").agg(
    F.count("order_id").alias("times_bought_together")
).filter(
    F.col("times_bought_together") >= expected_metrics["min_times_together"]
).withColumn(
    "_gold_timestamp", F.current_timestamp()
).orderBy(
    F.desc("times_bought_together")
)

print(f"  Filtered pairs (>= {expected_metrics['min_times_together']} occurrences): {product_pairs_gold.count():,}")
print("\nPreview - Top 10 Product Pairs:")
product_pairs_gold.show(10, truncate=False)

In [0]:
print_validation_header("product_pairs - Technical Validations")

# Initialize validation flag
validation_passed_technical = True

# 1. Row count check
pair_count = product_pairs_gold.count()
print(f"\nProduct pair count: {pair_count:,}")
print(f"Expected: >= {expected_metrics['min_pairs']:,}\n")

if pair_count >= expected_metrics["min_pairs"]:
    status = "PASS"
    msg = f"Pair count ({pair_count:,}) >= {expected_metrics['min_pairs']:,}"
else:
    status = "FAIL"
    msg = f"Pair count ({pair_count:,}) < {expected_metrics['min_pairs']:,}"
    validation_passed_technical = False
print_check_result("PAIR COUNT (>= 1M)", status, msg)

# 2. NOT NULL checks
print("\n2. NOT NULL Validations:")
critical_columns = ["product_id_1", "product_id_2", "times_bought_together"]
status, failed, msg = check_not_null(product_pairs_gold, critical_columns)
print_check_result(f"NOT NULL ({len(critical_columns)} columns)", status, msg, failed)
if status == "FAIL":
    validation_passed_technical = False

# 3. Filter threshold check
print("\n3. Filter Threshold Validation:")
below_threshold = product_pairs_gold.filter(
    F.col("times_bought_together") < expected_metrics["min_times_together"]
).count()

if below_threshold == 0:
    status = "PASS"
    msg = f"All pairs have times_bought_together >= {expected_metrics['min_times_together']}"
else:
    status = "FAIL"
    msg = f"{below_threshold} pairs below threshold ({expected_metrics['min_times_together']})"
    validation_passed_technical = False
print_check_result(f"FILTER THRESHOLD (>= {expected_metrics['min_times_together']})", status, msg, below_threshold)

print("\n" + "="*60)
if validation_passed_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("product_pairs - Business Validations")

# Initialize business validation flag
validation_passed_business = True

# 1. No reversed duplicates (product_id_1 < product_id_2)
print("\n1. Business Rule - No Reversed Duplicates:")
reversed_pairs = product_pairs_gold.filter(
    F.col("product_id_1") >= F.col("product_id_2")
).count()

if reversed_pairs == 0:
    status = "PASS"
    msg = "All pairs satisfy product_id_1 < product_id_2"
else:
    status = "FAIL"
    msg = f"{reversed_pairs} pairs have product_id_1 >= product_id_2"
    validation_passed_business = False
print_check_result("NO REVERSED DUPLICATES", status, msg, reversed_pairs)

# 2. Distribution analysis
print("\n2. Business Rule - Distribution Analysis:")
top_pairs = product_pairs_gold.limit(5).collect()
print("\n  Top 5 Product Pairs by Co-Purchase Frequency:")
for pair in top_pairs:
    print(f"    - Product {pair['product_id_1']} + Product {pair['product_id_2']}: {pair['times_bought_together']:,} times")

status = "PASS"
msg = f"Distribution looks reasonable ({pair_count:,} unique pairs)"
print_check_result("DISTRIBUTION ANALYSIS", status, msg)

print("\n" + "="*60)
if validation_passed_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results
validation_passed = validation_passed_technical and validation_passed_business

print("\n" + "="*60)
print("OVERALL VALIDATION RESULT")
print("="*60)
print(f"  Technical Validation: {'PASSED ✓' if validation_passed_technical else 'FAILED ✗'}")
print(f"  Business Validation:  {'PASSED ✓' if validation_passed_business else 'FAILED ✗'}")
print("="*60)

if validation_passed:
    print(f"\n✓ ALL VALIDATIONS PASSED - Ready to persist to Gold layer")
else:
    print(f"\n✗ SOME VALIDATIONS FAILED - Will NOT persist")
    print("\nPlease review and fix the errors above before re-running.")

print("="*60)

In [0]:
# Only persist if validation passed
if validation_passed:
    print("Persisting to Gold layer...\n")
    
    product_pairs_gold.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{gold_schema}.{target_table}")
    
    # Verify
    final_count = spark.table(f"{gold_schema}.{target_table}").count()
    
    print("\n" + "="*60)
    print("SUCCESS: Product Pairs table persisted to Gold layer")
    print("="*60)
    print(f"\nFinal Statistics:")
    print(f"  Table: {gold_schema}.{target_table}")
    print(f"  Rows: {final_count:,}")
    print(f"  Type: Product pair recommendations")
    print(f"  Format: Delta")
    print(f"\nNext Step: Use for recommendation systems and cross-selling")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")